In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, recall_score, f1_score

# ===== Load your encoded classification dataset =====
df = pd.read_csv("diagnosed_diabetes for classification.csv")  # change file if needed
target = "diagnosed_diabetes"

y = df[target].astype(int)
X = df.drop(columns=[target])

# ===== Pipeline: SelectKBest -> RandomForest =====
pipe = Pipeline([
    ("select", SelectKBest(score_func=mutual_info_classif)),
    ("rf", RandomForestClassifier(
        random_state=0,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

# ===== Grid over k and RF params =====
param_grid = {
    "select__k": [5, 8, 12, 16, 24, 32],  # adjust based on your column count
    "rf__n_estimators": [300, 500],
    "rf__max_depth": [None, 10, 20],
    "rf__min_samples_leaf": [1, 5, 10],
    "rf__max_features": ["sqrt", 0.5],
}

# For screening, recall_1 is often the key:
scoring = make_scorer(recall_score, pos_label=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring=scoring,     # swap to f1 scorer if you prefer
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("Best CV score (recall_1):", grid.best_score_)
print("Best params:", grid.best_params_)

# ===== Which features were selected? =====
best_pipe = grid.best_estimator_
mask = best_pipe.named_steps["select"].get_support()
selected_features = X.columns[mask].tolist()

print("\nSelected feature columns (encoded):")
print(selected_features)
print("\nNumber of selected columns:", len(selected_features))

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


/apps/python/3.10/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")
target = "diabetes_risk_score"

# avoid leakage if these exist
leak_cols = [c for c in ["diagnosed_diabetes", "diabetes_stage"] if c in df.columns]

y = df[target]
X = df.drop(columns=[target] + leak_cols)

pipe = Pipeline([
    ("select", SelectKBest(score_func=f_regression)),
    ("rf", RandomForestRegressor(random_state=0, n_jobs=-1))
])

param_grid = {
    "select__k": [5, 8, 12, 16, 24, 32],
    "rf__n_estimators": [300, 500],
    "rf__max_depth": [None, 10, 20],
    "rf__min_samples_leaf": [1, 5, 10],
    "rf__max_features": ["sqrt", 0.5],
}

cv = KFold(n_splits=5, shuffle=True, random_state=0)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",  # you can use "r2" too
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("Best CV RMSE:", -grid.best_score_)
print("Best params:", grid.best_params_)

best_pipe = grid.best_estimator_
mask = best_pipe.named_steps["select"].get_support()
selected = X.columns[mask].tolist()
print("\nSelected columns:", selected)
print("k =", len(selected))


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


/apps/python/3.10/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 

In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, recall_score

df = pd.read_csv("diagnosed_diabetes for classification.csv")
y = df["diagnosed_diabetes"].astype(int)
X = df.drop(columns=["diagnosed_diabetes"])

pipe = Pipeline([
    ("select", SelectKBest(mutual_info_classif)),
    ("rf", RandomForestClassifier(
        random_state=0, n_jobs=-1, class_weight="balanced_subsample"
    ))
])

param_distributions = {
    "select__k": [5, 8, 12, 16, 24, 32],
    "rf__n_estimators": [200, 400],          # keep small during search
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_leaf": [1, 2, 5, 10],
    "rf__max_features": ["sqrt", 0.5, 0.8],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)  # fewer folds during tuning
scoring = make_scorer(recall_score, pos_label=1)

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_distributions,
    n_iter=30,          # <- controls runtime
    scoring=scoring,
    cv=cv,
    n_jobs=-1,
    random_state=0,
    verbose=1
)

search.fit(X, y)
print("Best recall_1:", search.best_score_)
print("Best params:", search.best_params_)

# Final refit on ALL data with best settings (optionally increase trees)
best_model = search.best_estimator_
best_model.fit(X, y)


Fitting 3 folds for each of 30 candidates, totalling 90 fits


/apps/python/3.10/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/apps/python/3.10/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best recall_1: 0.8138605280264013
Best params: {'select__k': 24, 'rf__n_estimators': 400, 'rf__min_samples_leaf': 1, 'rf__max_features': 0.8, 'rf__max_depth': 30}


Pipeline(steps=[('select',
                 SelectKBest(k=24,
                             score_func=<function mutual_info_classif at 0x1498f0f67eb0>)),
                ('rf',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=30, max_features=0.8,
                                        n_estimators=400, n_jobs=-1,
                                        random_state=0))])

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, accuracy_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor


# =========================
# 1) Choose 6 non-invasive features (by name patterns)
# =========================
FEATURE_PATTERNS = [
    "age",
    "bmi",
    "systolic",        # or "sbp"
    "diastolic",       # or "dbp"
    "family_history",  # or "family"
    "physical_activity" # or "activity"
]

# Columns to NEVER use as features (targets/leakage-ish)
BLOCKLIST_SUBSTRINGS = [
    "diagnos", "diabetes_stage", "risk_score", "diabetes_risk", "stage",
    "hba1c", "glucose", "insulin"
]

RANDOM_STATE = 0
CV_SPLITS = 5


def choose_6_features(df, target_col, patterns=FEATURE_PATTERNS):
    cols = list(df.columns)

    # candidates = all columns except target and blocklist-ish
    def blocked(c):
        c_low = c.lower()
        if c == target_col:
            return True
        return any(b in c_low for b in BLOCKLIST_SUBSTRINGS)

    candidates = [c for c in cols if not blocked(c)]

    chosen = []
    for pat in patterns:
        pat_low = pat.lower()
        matches = [c for c in candidates if pat_low in c.lower()]
        if matches:
            # prefer shortest match (often the “base” column)
            best = sorted(matches, key=len)[0]
            if best not in chosen:
                chosen.append(best)

    # If we found fewer than 6, fill with other numeric columns (safe-ish)
    if len(chosen) < 6:
        numeric_candidates = [c for c in candidates if pd.api.types.is_numeric_dtype(df[c])]
        # avoid already chosen
        numeric_candidates = [c for c in numeric_candidates if c not in chosen]
        # heuristic: add higher-variance columns first
        numeric_candidates = sorted(
            numeric_candidates,
            key=lambda c: float(df[c].var()) if np.isfinite(df[c].var()) else -1.0,
            reverse=True
        )
        for c in numeric_candidates:
            chosen.append(c)
            if len(chosen) == 6:
                break

    return chosen[:6]


def rf_classification_report(df, target_col, dataset_name):
    y = df[target_col]
    # cast if looks like numeric classes
    if y.nunique() <= 20:
        y = y.astype(int)

    features = choose_6_features(df, target_col)
    X = df[features]

    print("\n" + "=" * 70)
    print(f"{dataset_name} (RF Classification)")
    print("Target:", target_col)
    print("Class counts:\n", y.value_counts())
    print("\nSelected 6 features:")
    for f in features:
        print(" -", f)

    rf = RandomForestClassifier(
        n_estimators=400,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample"
    )

    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    # OOF predictions for confusion matrix
    y_pred_oof = cross_val_predict(rf, X, y, cv=cv, method="predict", n_jobs=-1)

    labels = sorted(pd.unique(y))
    cm = confusion_matrix(y, y_pred_oof, labels=labels)

    acc = accuracy_score(y, y_pred_oof)

    if len(labels) == 2:
        rec1 = recall_score(y, y_pred_oof, pos_label=1, zero_division=0)
        f11 = f1_score(y, y_pred_oof, pos_label=1, zero_division=0)

        print("\nOOF Metrics:")
        print(f"- Accuracy: {acc:.3f}")
        print(f"- Recall_1: {rec1:.3f}")
        print(f"- F1_1:     {f11:.3f}")
    else:
        rec_macro = recall_score(y, y_pred_oof, average="macro", zero_division=0)
        f1_macro = f1_score(y, y_pred_oof, average="macro", zero_division=0)

        print("\nOOF Metrics:")
        print(f"- Accuracy:     {acc:.3f}")
        print(f"- Recall_macro: {rec_macro:.3f}")
        print(f"- F1_macro:     {f1_macro:.3f}")

    print("\nOOF Confusion Matrix:")
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels]
    )
    print(cm_df)


def rf_regression_report(df, target_col, dataset_name):
    y = df[target_col]
    features = choose_6_features(df, target_col)
    X = df[features]

    print("\n" + "=" * 70)
    print(f"{dataset_name} (RF Regression)")
    print("Target:", target_col)
    print("\nSelected 6 features:")
    for f in features:
        print(" -", f)

    rf = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    # OOF predictions
    y_pred_oof = cross_val_predict(rf, X, y, cv=cv, method="predict", n_jobs=-1)

    rmse = np.sqrt(mean_squared_error(y, y_pred_oof))
    mae = mean_absolute_error(y, y_pred_oof)
    r2 = r2_score(y, y_pred_oof)

    print("\nOOF Metrics:")
    print(f"- RMSE: {rmse:.3f}")
    print(f"- MAE:  {mae:.3f}")
    print(f"- R²:   {r2:.3f}")


# =========================
# 2) Run on your 3 datasets
# =========================

# A) diagnosed_diabetes (classification)
df_diag = pd.read_csv("diagnosed_diabetes for classification.csv")
rf_classification_report(df_diag, target_col="diagnosed_diabetes",
                         dataset_name="Case 1: Diagnosed Diabetes")

# B) diabetes_risk_score (regression)
df_risk = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")
# (optional) if these leakage columns exist, drop them before feature choice
for leak in ["diagnosed_diabetes", "diabetes_stage"]:
    if leak in df_risk.columns:
        df_risk = df_risk.drop(columns=[leak])
rf_regression_report(df_risk, target_col="diabetes_risk_score",
                     dataset_name="Case 2: Diabetes Risk Score")

# C) diabetes_stage (classification)
df_stage = pd.read_csv("prediabetes_for_classification.csv")
rf_classification_report(df_stage, target_col="diabetes_stage",
                         dataset_name="Case 3: Diabetes Stage")



Case 1: Diagnosed Diabetes (RF Classification)
Target: diagnosed_diabetes
Class counts:
 diagnosed_diabetes
1    59998
0    40002
Name: count, dtype: int64

Selected 6 features:
 - age
 - bmi
 - systolic_bp
 - diastolic_bp
 - family_history_diabetes
 - physical_activity_minutes_per_week

OOF Metrics:
- Accuracy: 0.608
- Recall_1: 0.777
- F1_1:     0.704

OOF Confusion Matrix:
        pred_0  pred_1
true_0   14180   25822
true_1   13372   46626

Case 2: Diabetes Risk Score (RF Regression)
Target: diabetes_risk_score

Selected 6 features:
 - age
 - bmi
 - systolic_bp
 - diastolic_bp
 - family_history_diabetes
 - physical_activity_minutes_per_week

OOF Metrics:
- RMSE: 1.689
- MAE:  1.347
- R²:   0.965

Case 3: Diabetes Stage (RF Classification)
Target: diabetes_stage
Class counts:
 diabetes_stage
1    31845
0     7981
Name: count, dtype: int64

Selected 6 features:
 - age
 - bmi
 - systolic_bp
 - diastolic_bp
 - family_history_diabetes
 - physical_activity_minutes_per_week

OOF Metrics:

In [5]:
import pandas as pd

df_diag = pd.read_csv("diagnosed_diabetes for classification.csv")
df_stage = pd.read_csv("prediabetes_for_classification.csv")

print("Diagnosed dataset rows:", df_diag.shape[0])
print("Stage dataset rows:    ", df_stage.shape[0])

print("\nDiagnosed target missing:", df_diag["diagnosed_diabetes"].isna().sum())
print("Stage target missing:   ", df_stage["diabetes_stage"].isna().sum())

print("\nDiagnosed class counts:\n", df_diag["diagnosed_diabetes"].value_counts(dropna=False))
print("\nStage class counts:\n", df_stage["diabetes_stage"].value_counts(dropna=False))
import pandas as pd

df = pd.read_csv("prediabetes_for_classification.csv")  # change file
y = df["diabetes_stage"]

counts = y.value_counts()
print(counts)
print("\nClass proportions:")
print((counts / counts.sum()).round(3))


Diagnosed dataset rows: 100000
Stage dataset rows:     39826

Diagnosed target missing: 0
Stage target missing:    0

Diagnosed class counts:
 diagnosed_diabetes
1    59998
0    40002
Name: count, dtype: int64

Stage class counts:
 diabetes_stage
1.0    31845
0.0     7981
Name: count, dtype: int64
diabetes_stage
1.0    31845
0.0     7981
Name: count, dtype: int64

Class proportions:
diabetes_stage
1.0    0.8
0.0    0.2
Name: count, dtype: float64


In [7]:
import pandas as pd

# ---------
# Settings
# ---------
FILE_DIAG = "diagnosed_diabetes for classification.csv"
TARGET_DIAG = "diagnosed_diabetes"

FILE_STAGE = "prediabetes_for_classification.csv"
TARGET_STAGE = "diabetes_stage"

FILE_RISK = "diabetes_preprocessed_for_risk_score.csv"
TARGET_RISK = "diabetes_risk_score"

RANDOM_STATE = 0

# Try imblearn; if missing, fallback to pandas resampling
try:
    from imblearn.over_sampling import RandomOverSampler
    IMBLEARN_OK = True
except Exception:
    IMBLEARN_OK = False


def balance_classification_ros(df, target_col, random_state=0):
    """Balance a classification dataset by RandomOverSampler (duplicates minority rows)."""
    y = df[target_col].astype(int)
    X = df.drop(columns=[target_col])

    print(f"\n[{target_col}] BEFORE counts:\n{y.value_counts()}")

    if IMBLEARN_OK:
        ros = RandomOverSampler(random_state=random_state)
        X_res, y_res = ros.fit_resample(X, y)
        out = pd.concat([pd.DataFrame(X_res, columns=X.columns),
                         pd.Series(y_res, name=target_col)], axis=1)
    else:
        # pandas fallback: oversample each class up to max count
        tmp = df.copy()
        tmp[target_col] = y
        max_n = tmp[target_col].value_counts().max()
        parts = []
        for cls, g in tmp.groupby(target_col):
            parts.append(g.sample(n=max_n, replace=True, random_state=random_state))
        out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=random_state).reset_index(drop=True)

    print(f"[{target_col}] AFTER counts:\n{out[target_col].value_counts()}")
    return out


def balance_regression_by_bins_ros(df, target_col, q=10, random_state=0):
    """
    Optional: "balance" regression by oversampling underrepresented y-ranges.
    We bin y into q-quantile bins, then RandomOverSample bins.
    """
    y = df[target_col]
    X = df.drop(columns=[target_col])

    bins = pd.qcut(y, q=q, duplicates="drop")
    bin_codes = bins.cat.codes

    print(f"\n[{target_col} regression] BEFORE bin counts (q={q}):\n{pd.Series(bin_codes).value_counts().sort_index()}")

    if IMBLEARN_OK:
        ros = RandomOverSampler(random_state=random_state)
        X_res, bin_res = ros.fit_resample(X, bin_codes)
        # recover y by duplicating rows from original df via indices approach:
        # easiest: rebuild via DataFrame and merge y from original row order using a temp index
        # Instead: do oversampling on the full df with bins as labels (pandas fallback style) if you want exact row duplication
        # Here’s a safe “exact duplicate rows” approach using pandas:
        tmp = df.copy()
        tmp["_bin"] = bin_codes
        max_n = tmp["_bin"].value_counts().max()
        parts = []
        for b, g in tmp.groupby("_bin"):
            parts.append(g.sample(n=max_n, replace=True, random_state=random_state))
        out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=random_state).drop(columns=["_bin"]).reset_index(drop=True)
    else:
        tmp = df.copy()
        tmp["_bin"] = bin_codes
        max_n = tmp["_bin"].value_counts().max()
        parts = []
        for b, g in tmp.groupby("_bin"):
            parts.append(g.sample(n=max_n, replace=True, random_state=random_state))
        out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=random_state).drop(columns=["_bin"]).reset_index(drop=True)

    # show after
    bins_after = pd.qcut(out[target_col], q=q, duplicates="drop").cat.codes
    print(f"[{target_col} regression] AFTER bin counts:\n{pd.Series(bins_after).value_counts().sort_index()}")
    return out


# =========================
# 1) Balance diagnosed_diabetes (classification)
# =========================
df_diag = pd.read_csv(FILE_DIAG)
df_diag_bal = balance_classification_ros(df_diag, TARGET_DIAG, random_state=RANDOM_STATE)
out_diag = "diagnosed_diabetes_for_classification_BALANCED_ROS.csv"
df_diag_bal.to_csv(out_diag, index=False)
print("Saved:", out_diag)

# =========================
# 2) Balance diabetes_stage (classification)
# =========================
df_stage = pd.read_csv(FILE_STAGE)
df_stage_bal = balance_classification_ros(df_stage, TARGET_STAGE, random_state=RANDOM_STATE)
out_stage = "prediabetes_for_classification_BALANCED_ROS.csv"
df_stage_bal.to_csv(out_stage, index=False)
print("Saved:", out_stage)

# =========================
# 3) Risk score (regression) — optional “bin-balanced” dataset
# =========================
df_risk = pd.read_csv(FILE_RISK)

# Drop leakage columns if present (recommended)
for leak in ["diagnosed_diabetes", "diabetes_stage"]:
    if leak in df_risk.columns:
        df_risk = df_risk.drop(columns=[leak])

df_risk_bal = balance_regression_by_bins_ros(df_risk, TARGET_RISK, q=10, random_state=RANDOM_STATE)
out_risk = "diabetes_preprocessed_for_risk_score_BIN_BALANCED.csv"
df_risk_bal.to_csv(out_risk, index=False)
print("Saved:", out_risk)



[diagnosed_diabetes] BEFORE counts:
diagnosed_diabetes
1    59998
0    40002
Name: count, dtype: int64
[diagnosed_diabetes] AFTER counts:
diagnosed_diabetes
1    59998
0    59998
Name: count, dtype: int64
Saved: diagnosed_diabetes_for_classification_BALANCED_ROS.csv

[diabetes_stage] BEFORE counts:
diabetes_stage
1    31845
0     7981
Name: count, dtype: int64
[diabetes_stage] AFTER counts:
diabetes_stage
0    31845
1    31845
Name: count, dtype: int64
Saved: prediabetes_for_classification_BALANCED_ROS.csv

[diabetes_risk_score regression] BEFORE bin counts (q=10):
0    10090
1    10106
2     9987
3    10247
4     9979
5     9989
6     9620
7     9991
8    10057
9     9934
Name: count, dtype: int64
[diabetes_risk_score regression] AFTER bin counts:
0    10247
1    10247
2    10247
3    10247
4    10247
5    10247
6    10247
7    10247
8    10247
9    10247
Name: count, dtype: int64
Saved: diabetes_preprocessed_for_risk_score_BIN_BALANCED.csv


In [8]:
# ==========================================================
# CASE 1: diagnosed_diabetes (Classification)
# Compare: (A) unbalanced RF, (B) class_weight RF, (C) oversample RF
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

RANDOM_STATE = 0
CV_SPLITS = 5

try:
    from imblearn.over_sampling import RandomOverSampler
    IMBLEARN_OK = True
except Exception:
    IMBLEARN_OK = False


def _ros_resample(X_tr: pd.DataFrame, y_tr: pd.Series):
    if IMBLEARN_OK:
        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_res, y_res = ros.fit_resample(X_tr, y_tr)
        return pd.DataFrame(X_res, columns=X_tr.columns), pd.Series(y_res, name=y_tr.name)

    tmp = X_tr.copy()
    tmp["_y_"] = y_tr.values
    max_n = tmp["_y_"].value_counts().max()
    parts = []
    for cls, g in tmp.groupby("_y_"):
        parts.append(g.sample(n=max_n, replace=True, random_state=RANDOM_STATE))
    out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    y_res = out["_y_"].astype(int)
    X_res = out.drop(columns=["_y_"])
    return X_res, y_res


def evaluate_rf_binary(X, y, mode="none"):
    if mode == "class_weight":
        base = RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1,
            class_weight="balanced_subsample"
        )
    else:
        base = RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1
        )

    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    y_true_all, y_pred_all = [], []
    accs, precs, recs, f1s = [], [], [], []

    for tr, te in cv.split(X, y):
        m = clone(base)
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        if mode == "oversample":
            X_tr, y_tr = _ros_resample(X_tr, y_tr)

        m.fit(X_tr, y_tr)
        pred = m.predict(X_te)

        y_true_all.append(y_te.to_numpy())
        y_pred_all.append(pred)

        accs.append(accuracy_score(y_te, pred))
        precs.append(precision_score(y_te, pred, pos_label=1, zero_division=0))
        recs.append(recall_score(y_te, pred, pos_label=1, zero_division=0))
        f1s.append(f1_score(y_te, pred, pos_label=1, zero_division=0))

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])

    return {
        "mode": mode,
        "accuracy": (float(np.mean(accs)), float(np.std(accs))),
        "precision_1": (float(np.mean(precs)), float(np.std(precs))),
        "recall_1": (float(np.mean(recs)), float(np.std(recs))),
        "f1_1": (float(np.mean(f1s)), float(np.std(f1s))),
        "cm": cm
    }


def show_results(res):
    print(f"\n=== RF mode: {res['mode']} ===")
    print(f"Accuracy:   {res['accuracy'][0]:.3f} ± {res['accuracy'][1]:.3f}")
    print(f"Precision1: {res['precision_1'][0]:.3f} ± {res['precision_1'][1]:.3f}")
    print(f"Recall1:    {res['recall_1'][0]:.3f} ± {res['recall_1'][1]:.3f}")
    print(f"F1_1:       {res['f1_1'][0]:.3f} ± {res['f1_1'][1]:.3f}")
    print("OOF Confusion matrix:")
    print(pd.DataFrame(res["cm"], index=["true_0","true_1"], columns=["pred_0","pred_1"]))


# ---- Load dataset
df = pd.read_csv("diagnosed_diabetes for classification.csv")
y = df["diagnosed_diabetes"].astype(int)
X = df.drop(columns=["diagnosed_diabetes"])

print("CASE 1: diagnosed_diabetes")
print("Rows:", len(df))
print("Class counts:\n", y.value_counts())

# ---- Run 3 modes
for mode in ["none", "class_weight", "oversample"]:
    show_results(evaluate_rf_binary(X, y, mode=mode))


CASE 1: diagnosed_diabetes
Rows: 100000
Class counts:
 diagnosed_diabetes
1    59998
0    40002
Name: count, dtype: int64

=== RF mode: none ===
Accuracy:   0.626 ± 0.002
Precision1: 0.649 ± 0.001
Recall1:    0.823 ± 0.003
F1_1:       0.726 ± 0.002
OOF Confusion matrix:
        pred_0  pred_1
true_0   13271   26731
true_1   10626   49372

=== RF mode: class_weight ===
Accuracy:   0.626 ± 0.001
Precision1: 0.644 ± 0.001
Recall1:    0.842 ± 0.003
F1_1:       0.730 ± 0.001
OOF Confusion matrix:
        pred_0  pred_1
true_0   12051   27951
true_1    9493   50505

=== RF mode: oversample ===
Accuracy:   0.615 ± 0.003
Precision1: 0.685 ± 0.002
Recall1:    0.663 ± 0.003
F1_1:       0.674 ± 0.002
OOF Confusion matrix:
        pred_0  pred_1
true_0   21729   18273
true_1   20235   39763


In [1]:
# ==========================================================
# CASE 2: diabetes_risk_score (Regression)
# Compare: (A) normal RF regression, (B) optional bin-balanced RF regression
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone

RANDOM_STATE = 0
CV_SPLITS = 5

def _bin_balance_regression(X_tr: pd.DataFrame, y_tr: pd.Series, q=10):
    tmp = X_tr.copy()
    tmp["_y_"] = y_tr.values
    tmp["_bin_"] = pd.qcut(tmp["_y_"], q=q, duplicates="drop").cat.codes

    max_n = tmp["_bin_"].value_counts().max()
    parts = []
    for b, g in tmp.groupby("_bin_"):
        parts.append(g.sample(n=max_n, replace=True, random_state=RANDOM_STATE))
    out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    y_res = out["_y_"]
    X_res = out.drop(columns=["_y_", "_bin_"])
    return X_res, y_res

def evaluate_rf_regression(X, y, mode="none"):
    base = RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)
    cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    rmses, maes, r2s = [], [], []

    for tr, te in cv.split(X):
        m = clone(base)
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        if mode == "bin_balanced":
            X_tr, y_tr = _bin_balance_regression(X_tr, y_tr, q=10)

        m.fit(X_tr, y_tr)
        pred = m.predict(X_te)

        rmses.append(np.sqrt(mean_squared_error(y_te, pred)))
        maes.append(mean_absolute_error(y_te, pred))
        r2s.append(r2_score(y_te, pred))

    return {
        "mode": mode,
        "rmse": (float(np.mean(rmses)), float(np.std(rmses))),
        "mae":  (float(np.mean(maes)), float(np.std(maes))),
        "r2":   (float(np.mean(r2s)), float(np.std(r2s))),
    }

def show_results(res):
    print(f"\n=== RF Regression mode: {res['mode']} ===")
    print(f"RMSE: {res['rmse'][0]:.3f} ± {res['rmse'][1]:.3f}")
    print(f"MAE:  {res['mae'][0]:.3f} ± {res['mae'][1]:.3f}")
    print(f"R²:   {res['r2'][0]:.3f} ± {res['r2'][1]:.3f}")

# ---- Load dataset
df = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")

# prevent leakage if present
for leak in ["diagnosed_diabetes", "diabetes_stage"]:
    if leak in df.columns:
        df = df.drop(columns=[leak])

y = df["diabetes_risk_score"]
X = df.drop(columns=["diabetes_risk_score"])

print("CASE 2: diabetes_risk_score (Regression)")
print("Rows:", len(df))

# ---- Compare
for mode in ["none", "bin_balanced"]:
    show_results(evaluate_rf_regression(X, y, mode=mode))


CASE 2: diabetes_risk_score (Regression)
Rows: 100000

=== RF Regression mode: none ===
RMSE: 1.283 ± 0.005
MAE:  1.026 ± 0.005
R²:   0.980 ± 0.000

=== RF Regression mode: bin_balanced ===
RMSE: 1.329 ± 0.007
MAE:  1.060 ± 0.007
R²:   0.978 ± 0.000


In [5]:
# ==========================================================
# CASE 1 — Random Forest on diagnosed_diabetes (Classification)
# Compare: non_balanced vs class_weight_balanced vs oversample_balanced
# Outputs: mean±std metrics + OOF confusion matrix for each mode
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 0
CV_SPLITS = 5

try:
    from imblearn.over_sampling import RandomOverSampler
    IMBLEARN_OK = True
except Exception:
    IMBLEARN_OK = False


def to_numeric_X(df):
    X = pd.get_dummies(df, drop_first=False)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    return X


def oversample_train_fold(X_tr, y_tr):
    if IMBLEARN_OK:
        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_res, y_res = ros.fit_resample(X_tr, y_tr)
        return pd.DataFrame(X_res, columns=X_tr.columns), pd.Series(y_res, name=y_tr.name)

    tmp = X_tr.copy()
    tmp["_y_"] = y_tr.values
    max_n = tmp["_y_"].value_counts().max()
    parts = []
    for cls, g in tmp.groupby("_y_"):
        parts.append(g.sample(n=max_n, replace=True, random_state=RANDOM_STATE))
    out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return out.drop(columns=["_y_"]), out["_y_"]


def mean_std(a):
    a = np.asarray(a, dtype=float)
    return float(a.mean()), float(a.std())


def run_case1():
    df = pd.read_csv("diagnosed_diabetes for classification.csv")
    y = df["diagnosed_diabetes"].astype(int)
    X = to_numeric_X(df.drop(columns=["diagnosed_diabetes"]))

    print("CASE 1: diagnosed_diabetes")
    print("Rows:", len(df))
    print("Class counts:\n", y.value_counts())

    modes = {
        "non_balanced": RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
        "class_weight_balanced": RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced_subsample"
        ),
        "oversample_balanced": RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    }

    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for mode, base_model in modes.items():
        accs, precs, recs, f1s = [], [], [], []
        y_true_all, y_pred_all = [], []

        for tr, te in cv.split(X, y):
            m = clone(base_model)
            X_tr, X_te = X.iloc[tr], X.iloc[te]
            y_tr, y_te = y.iloc[tr], y.iloc[te]

            if mode == "oversample_balanced":
                X_tr, y_tr = oversample_train_fold(X_tr, y_tr)

            m.fit(X_tr, y_tr)
            pred = m.predict(X_te)

            y_true_all.append(y_te.to_numpy())
            y_pred_all.append(pred)

            accs.append(accuracy_score(y_te, pred))
            precs.append(precision_score(y_te, pred, pos_label=1, zero_division=0))
            recs.append(recall_score(y_te, pred, pos_label=1, zero_division=0))
            f1s.append(f1_score(y_te, pred, pos_label=1, zero_division=0))

        y_true_all = np.concatenate(y_true_all)
        y_pred_all = np.concatenate(y_pred_all)
        cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])

        acc_m, acc_s = mean_std(accs)
        pre_m, pre_s = mean_std(precs)
        rec_m, rec_s = mean_std(recs)
        f1_m,  f1_s  = mean_std(f1s)

        print("\n---", mode, "---")
        print(f"Accuracy:   {acc_m:.3f} ± {acc_s:.3f}")
        print(f"Precision1: {pre_m:.3f} ± {pre_s:.3f}")
        print(f"Recall1:    {rec_m:.3f} ± {rec_s:.3f}")
        print(f"F1_1:       {f1_m:.3f} ± {f1_s:.3f}")
        print("OOF Confusion matrix:")
        print(pd.DataFrame(cm, index=["true_0","true_1"], columns=["pred_0","pred_1"]))


run_case1()


CASE 1: diagnosed_diabetes
Rows: 100000
Class counts:
 diagnosed_diabetes
1    59998
0    40002
Name: count, dtype: int64

--- non_balanced ---
Accuracy:   0.626 ± 0.002
Precision1: 0.649 ± 0.001
Recall1:    0.823 ± 0.003
F1_1:       0.726 ± 0.002
OOF Confusion matrix:
        pred_0  pred_1
true_0   13271   26731
true_1   10626   49372

--- class_weight_balanced ---
Accuracy:   0.626 ± 0.001
Precision1: 0.644 ± 0.001
Recall1:    0.842 ± 0.003
F1_1:       0.730 ± 0.001
OOF Confusion matrix:
        pred_0  pred_1
true_0   12051   27951
true_1    9493   50505

--- oversample_balanced ---
Accuracy:   0.615 ± 0.003
Precision1: 0.685 ± 0.002
Recall1:    0.663 ± 0.003
F1_1:       0.674 ± 0.002
OOF Confusion matrix:
        pred_0  pred_1
true_0   21729   18273
true_1   20235   39763


In [6]:
# ==========================================================
# CASE 2 — Random Forest on diabetes_risk_score (Regression)
# Compare: non_balanced vs bin_balanced_optional
# Outputs: mean±std RMSE/MAE/R2 for each mode
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 0
CV_SPLITS = 5


def to_numeric_X(df):
    X = pd.get_dummies(df, drop_first=False)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    return X


def mean_std(a):
    a = np.asarray(a, dtype=float)
    return float(a.mean()), float(a.std())


def bin_balance_train_fold(X_tr, y_tr, q=10):
    tmp = X_tr.copy()
    tmp["_y_"] = y_tr.values
    tmp["_bin_"] = pd.qcut(tmp["_y_"], q=q, duplicates="drop").cat.codes

    max_n = tmp["_bin_"].value_counts().max()
    parts = []
    for b, g in tmp.groupby("_bin_"):
        parts.append(g.sample(n=max_n, replace=True, random_state=RANDOM_STATE))
    out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return out.drop(columns=["_y_", "_bin_"]), out["_y_"]


def run_case2():
    df = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")

    # Drop leakage columns if present
    for leak in ["diagnosed_diabetes", "diabetes_stage"]:
        if leak in df.columns:
            df = df.drop(columns=[leak])

    y = df["diabetes_risk_score"]
    X = to_numeric_X(df.drop(columns=["diabetes_risk_score"]))

    print("CASE 2: diabetes_risk_score (Regression)")
    print("Rows:", len(df))

    modes = ["non_balanced", "bin_balanced_optional"]
    base = RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)
    cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for mode in modes:
        rmses, maes, r2s = [], [], []

        for tr, te in cv.split(X):
            m = clone(base)
            X_tr, X_te = X.iloc[tr], X.iloc[te]
            y_tr, y_te = y.iloc[tr], y.iloc[te]

            if mode == "bin_balanced_optional":
                X_tr, y_tr = bin_balance_train_fold(X_tr, y_tr, q=10)

            m.fit(X_tr, y_tr)
            pred = m.predict(X_te)

            rmses.append(np.sqrt(mean_squared_error(y_te, pred)))
            maes.append(mean_absolute_error(y_te, pred))
            r2s.append(r2_score(y_te, pred))

        rm_m, rm_s = mean_std(rmses)
        ma_m, ma_s = mean_std(maes)
        r2_m, r2_s = mean_std(r2s)

        print("\n---", mode, "---")
        print(f"RMSE: {rm_m:.3f} ± {rm_s:.3f}")
        print(f"MAE:  {ma_m:.3f} ± {ma_s:.3f}")
        print(f"R²:   {r2_m:.3f} ± {r2_s:.3f}")


run_case2()


CASE 2: diabetes_risk_score (Regression)
Rows: 100000

--- non_balanced ---
RMSE: 1.283 ± 0.005
MAE:  1.026 ± 0.005
R²:   0.980 ± 0.000

--- bin_balanced_optional ---
RMSE: 1.329 ± 0.007
MAE:  1.060 ± 0.007
R²:   0.978 ± 0.000


In [13]:
df = pd.read_csv("prediabetes_for_classification.csv")
column=df['diabetes_stage']
column.head()
# df.head()

0    0.0
1    1.0
2    1.0
3    0.0
4    0.0
Name: diabetes_stage, dtype: float64

In [7]:
# ==========================================================
# CASE 3 — Random Forest on diabetes_stage (Classification; binary or multiclass)
# Compare: non_balanced vs class_weight_balanced vs oversample_balanced
# Outputs:
#   - Binary: Recall1/F1_1
#   - Multiclass: MacroRecall/MacroF1
# plus OOF confusion matrix for each mode
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 0
CV_SPLITS = 5

try:
    from imblearn.over_sampling import RandomOverSampler
    IMBLEARN_OK = True
except Exception:
    IMBLEARN_OK = False


def to_numeric_X(df):
    X = pd.get_dummies(df, drop_first=False)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    return X


def oversample_train_fold(X_tr, y_tr):
    if IMBLEARN_OK:
        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_res, y_res = ros.fit_resample(X_tr, y_tr)
        return pd.DataFrame(X_res, columns=X_tr.columns), pd.Series(y_res, name=y_tr.name)

    tmp = X_tr.copy()
    tmp["_y_"] = y_tr.values
    max_n = tmp["_y_"].value_counts().max()
    parts = []
    for cls, g in tmp.groupby("_y_"):
        parts.append(g.sample(n=max_n, replace=True, random_state=RANDOM_STATE))
    out = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return out.drop(columns=["_y_"]), out["_y_"]


def mean_std(a):
    a = np.asarray(a, dtype=float)
    return float(a.mean()), float(a.std())


def run_case3():
    df = pd.read_csv("prediabetes_for_classification.csv")
    y = df["diabetes_stage"]
    if y.nunique() <= 50:
        y = y.astype(int)
    X = to_numeric_X(df.drop(columns=["diabetes_stage"]))

    labels = sorted(pd.unique(y))
    multiclass = (len(labels) > 2)

    print("CASE 3: diabetes_stage")
    print("Rows:", len(df))
    print("Class counts:\n", y.value_counts())
    print("Multiclass:", multiclass)

    modes = {
        "non_balanced": RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
        "class_weight_balanced": RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced_subsample"
        ),
        "oversample_balanced": RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    }

    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for mode, base_model in modes.items():
        accs, precs, recs, f1s = [], [], [], []
        y_true_all, y_pred_all = [], []

        for tr, te in cv.split(X, y):
            m = clone(base_model)
            X_tr, X_te = X.iloc[tr], X.iloc[te]
            y_tr, y_te = y.iloc[tr], y.iloc[te]

            if mode == "oversample_balanced":
                X_tr, y_tr = oversample_train_fold(X_tr, y_tr)

            m.fit(X_tr, y_tr)
            pred = m.predict(X_te)

            y_true_all.append(y_te.to_numpy())
            y_pred_all.append(pred)

            accs.append(accuracy_score(y_te, pred))

            if multiclass:
                precs.append(precision_score(y_te, pred, average="macro", zero_division=0))
                recs.append(recall_score(y_te, pred, average="macro", zero_division=0))
                f1s.append(f1_score(y_te, pred, average="macro", zero_division=0))
            else:
                precs.append(precision_score(y_te, pred, pos_label=1, zero_division=0))
                recs.append(recall_score(y_te, pred, pos_label=1, zero_division=0))
                f1s.append(f1_score(y_te, pred, pos_label=1, zero_division=0))

        y_true_all = np.concatenate(y_true_all)
        y_pred_all = np.concatenate(y_pred_all)
        cm = confusion_matrix(y_true_all, y_pred_all, labels=labels)

        acc_m, acc_s = mean_std(accs)
        pre_m, pre_s = mean_std(precs)
        rec_m, rec_s = mean_std(recs)
        f1_m,  f1_s  = mean_std(f1s)

        print("\n---", mode, "---")
        print(f"Accuracy: {acc_m:.3f} ± {acc_s:.3f}")
        if multiclass:
            print(f"Precision_macro: {pre_m:.3f} ± {pre_s:.3f}")
            print(f"Recall_macro:    {rec_m:.3f} ± {rec_s:.3f}")
            print(f"F1_macro:        {f1_m:.3f} ± {f1_s:.3f}")
        else:
            print(f"Precision1: {pre_m:.3f} ± {pre_s:.3f}")
            print(f"Recall1:    {rec_m:.3f} ± {rec_s:.3f}")
            print(f"F1_1:       {f1_m:.3f} ± {f1_s:.3f}")

        print("OOF Confusion matrix:")
        print(pd.DataFrame(cm,
                           index=[f"true_{l}" for l in labels],
                           columns=[f"pred_{l}" for l in labels]))


run_case3()



CASE 3: diabetes_stage
Rows: 39826
Class counts:
 diabetes_stage
1    31845
0     7981
Name: count, dtype: int64
Multiclass: False

--- non_balanced ---
Accuracy: 0.799 ± 0.000
Precision1: 0.800 ± 0.000
Recall1:    1.000 ± 0.000
F1_1:       0.889 ± 0.000
OOF Confusion matrix:
        pred_0  pred_1
true_0       6    7975
true_1      14   31831

--- class_weight_balanced ---
Accuracy: 0.799 ± 0.000
Precision1: 0.800 ± 0.000
Recall1:    1.000 ± 0.000
F1_1:       0.889 ± 0.000
OOF Confusion matrix:
        pred_0  pred_1
true_0       1    7980
true_1       7   31838

--- oversample_balanced ---
Accuracy: 0.785 ± 0.002
Precision1: 0.806 ± 0.001
Recall1:    0.964 ± 0.004
F1_1:       0.878 ± 0.001
OOF Confusion matrix:
        pred_0  pred_1
true_0     579    7402
true_1    1145   30700
